# 05 — Fine tuning (TensorFlow)

**Qué hace:** dos rondas de fine-tuning sobre modelos Keras ya entrenados: se descongela el último 20% de capas del backbone MobileNetV3Small, se recompila con un learning rate bajo (`Adam`, `1e-4`) y se reentrena unas pocas épocas. Cada ronda incluye evaluación completa (matriz de confusión en texto, `classification_report`, aciertos por clase, tiempo de inferencia).

**Consume:** `model_my_tf.keras` (primera ronda — modelo pre-entrenado que debe existir de antemano; el notebook original tampoco muestra dónde se genera ese archivo exacto, es una dependencia externa preexistente) y `model_my_tf_fine_tuned_.keras` (segunda ronda, producido por `04_transfer_learning.ipynb`). Ambas rondas usan `asl_split_nuevo/train|test|validate/<clase>/*`.

**Produce:** `finetunned.keras` (modelo final tras la segunda ronda de fine-tuning, consumido luego por `07_hibrido_landmarks.ipynb`).

## Ronda 1 — FINE TUNING TENSOR FLOW: carga del modelo y descongelado del 20% final

In [13]:
import tensorflow as tf
from tensorflow.keras.utils import image_dataset_from_directory

In [3]:
model_cargado = tf.keras.models.load_model('model_my_tf.keras')

/Users/denilsonbarahona/Documents/IA/model-sign-language/.venv/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 6 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [5]:
print(f"MobileNetV3 tiene {len(model_cargado.layers)} capas")

for i, capa in enumerate(model_cargado.layers):
    print(i, capa.name, type(capa).__name__)

MobileNetV3 tiene 6 capas
0 input_layer_12 InputLayer
1 MobileNetV3Small Functional
2 global_average_pooling2d_3 GlobalAveragePooling2D
3 dense_6 Dense
4 dropout_3 Dropout
5 dense_7 Dense


In [9]:
base = model_cargado.layers[1]   # ajusta el índice al que viste en el print
print(f"La base tiene {len(base.layers)} capas internas")

La base tiene 157 capas internas


In [10]:
base.trainable = True

total_layers = len(base.layers)
limit = int(total_layers * 0.80)

for layer in base.layers[:limit]:
    layer.trainable = False

print(f"Congeladas: {limit} | Entrenables: {total_layers - limit}")


Congeladas: 125 | Entrenables: 32


## Recompilación con learning rate bajo

In [11]:
model_cargado.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy', 'precision', 'recall']
)



In [12]:
model_cargado.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_12 (InputLayer)     │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MobileNetV3Small (Functional)   │ (None, 7, 7, 576)      │       939,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 576)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 224)            │       129,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 224)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 36)             │         8,100 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,076,468 (4.11 MB)

 Trainable params: 726,692 (2.77 MB)

 Non-trainable params: 349,776 (1.33 MB)

## Carga de datos y entrenamiento

In [15]:
from pathlib import Path
DESTINO = Path('./asl_split_nuevo')   
train_dir = DESTINO / 'train'
test_dir = DESTINO / 'test'
val_dir = DESTINO / 'validate'



trained_generated = image_dataset_from_directory(
    train_dir,
    labels='inferred',
    batch_size = 100,
    color_mode='rgb',
    image_size=(224, 224),
    label_mode="categorical",
    pad_to_aspect_ratio=True,
)

test_generated = image_dataset_from_directory(
    test_dir,
    labels='inferred',
    batch_size = 100,
    color_mode='rgb',
    image_size=(224, 224),
    label_mode="categorical",
    pad_to_aspect_ratio=True,
)

val_generated = image_dataset_from_directory(
    val_dir,
    labels='inferred',
    batch_size = 100,
    color_mode='rgb',
    image_size=(224, 224),
    label_mode="categorical",
    pad_to_aspect_ratio=True,
)

Found 8620 files belonging to 36 classes.
Found 14517 files belonging to 36 classes.
Found 12267 files belonging to 36 classes.


In [16]:
earlyStopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True 
)

In [17]:
history_ft = model_cargado.fit(
    trained_generated,
    epochs=15,
    validation_data=val_generated,
    callbacks=[earlyStopping]
)

Epoch 1/15


/Users/denilsonbarahona/Documents/IA/model-sign-language/.venv/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


87/87 ━━━━━━━━━━━━━━━━━━━━ 140s 2s/step - accuracy: 0.8875 - loss: 0.5277 - precision: 0.9075 - recall: 0.8780 - val_accuracy: 0.7472 - val_loss: 1.0407 - val_precision: 0.7840 - val_recall: 0.7221
Epoch 2/15
87/87 ━━━━━━━━━━━━━━━━━━━━ 122s 1s/step - accuracy: 0.9885 - loss: 0.0440 - precision: 0.9910 - recall: 0.9870 - val_accuracy: 0.7429 - val_loss: 1.0314 - val_precision: 0.7756 - val_recall: 0.7177
Epoch 3/15
87/87 ━━━━━━━━━━━━━━━━━━━━ 118s 1s/step - accuracy: 0.9943 - loss: 0.0218 - precision: 0.9952 - recall: 0.9929 - val_accuracy: 0.7638 - val_loss: 0.9662 - val_precision: 0.7918 - val_recall: 0.7430
Epoch 4/15
87/87 ━━━━━━━━━━━━━━━━━━━━ 118s 1s/step - accuracy: 0.9964 - loss: 0.0158 - precision: 0.9971 - recall: 0.9956 - val_accuracy: 0.7788 - val_loss: 0.9252 - val_precision: 0.8031 - val_recall: 0.7609
Epoch 5/15
87/87 ━━━━━━━━━━━━━━━━━━━━ 119s 1s/step - accuracy: 0.9984 - loss: 0.0089 - precision: 0.9986 - recall: 0.9980 - val_accuracy: 0.7832 - val_loss: 0.8876 - val_preci

In [18]:
model_cargado.evaluate(test_generated)

146/146 ━━━━━━━━━━━━━━━━━━━━ 81s 555ms/step - accuracy: 0.8469 - loss: 0.6210 - precision: 0.8637 - recall: 0.8388


[0.6209959387779236, 0.8469380736351013, 0.863739550113678, 0.8388096690177917]

## Evaluación: matriz de confusión, classification report, aciertos por clase, tiempo de inferencia

In [19]:
import numpy as np

y_true, y_pred = [], []

for imagenes, etiquetas in test_generated:
    salidas = model_cargado.predict(imagenes, verbose=0)
    y_pred.extend(np.argmax(salidas, axis=1))     # clase predicha
    y_true.extend(np.argmax(etiquetas.numpy(), axis=1))  # one-hot -> índice

y_pred = np.array(y_pred)
y_true = np.array(y_true)
nombres_clases = test_generated.class_names
print(f"{len(y_pred)} predicciones recolectadas")

14517 predicciones recolectadas


In [24]:
from sklearn.metrics import confusion_matrix

matriz = confusion_matrix(y_true, y_pred)
num_clases = len(nombres_clases)

# Ancho de cada celda (sube a 5 si algún conteo pasa de 999)
w = 4

# Encabezado: nombres de las clases predichas, arriba
encabezado = " " * 6 + "".join(f"{c:>{w}}" for c in nombres_clases)
print("Predicho →")
print(encabezado)
print("Real ↓")

# Cada fila: nombre de la clase real + sus conteos
for i, clase_real in enumerate(nombres_clases):
    fila = f"{clase_real:>5} " + "".join(f"{int(matriz[i, j]):>{w}}" for j in range(num_clases))
    print(fila)

Predicho →
         0   1   2   3   4   5   6   7   8   9   A   B   C   D   E   F   G   H   I   J   K   L   M   N   O   P   Q   R   S   T   U   V   W   X   Y   Z
Real ↓
    0   60   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    1    0 368   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    2    0  98 303   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   2   1   0   0   0
    3    0   0   0 358   0   0   0  18  10   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    4    0   0   0   0 415   0   4   0   0  11   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    5    0   0   0   0  46 446   0   0   0  36   0   0   0   0   0   0   0  

In [21]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred, target_names=nombres_clases, digits=3))

              precision    recall  f1-score   support

           0      1.000     1.000     1.000        60
           1      0.471     1.000     0.641       368
           2      0.454     0.750     0.566       404
           3      0.940     0.927     0.934       386
           4      0.859     0.965     0.909       430
           5      1.000     0.845     0.916       528
           6      0.752     0.705     0.728       413
           7      0.808     0.409     0.543       474
           8      0.627     0.901     0.739       382
           9      0.907     0.994     0.949       464
           A      0.987     1.000     0.993       453
           B      0.813     1.000     0.897       425
           C      1.000     0.845     0.916       453
           D      0.992     0.575     0.729       457
           E      0.815     0.773     0.793       427
           F      0.996     0.547     0.707       495
           G      1.000     0.761     0.864       448
           H      0.976    

In [22]:
print("ACIERTOS por clase:")
for i, clase in enumerate(nombres_clases):
    total = int(matriz[i, :].sum())
    ok = int(matriz[i, i])
    print(f"{clase:>3}: {ok:>3}/{total:<3} ({ok/total*100:.0f}%)")

ACIERTOS por clase:
  0:  60/60  (100%)
  1: 368/368 (100%)
  2: 303/404 (75%)
  3: 358/386 (93%)
  4: 415/430 (97%)
  5: 446/528 (84%)
  6: 291/413 (70%)
  7: 194/474 (41%)
  8: 344/382 (90%)
  9: 461/464 (99%)
  A: 453/453 (100%)
  B: 425/425 (100%)
  C: 383/453 (85%)
  D: 263/457 (58%)
  E: 330/427 (77%)
  F: 271/495 (55%)
  G: 341/448 (76%)
  H: 286/286 (100%)
  I: 376/479 (78%)
  J: 509/523 (97%)
  K: 331/346 (96%)
  L: 366/366 (100%)
  M: 330/330 (100%)
  N: 301/368 (82%)
  O: 454/480 (95%)
  P: 279/279 (100%)
  Q: 304/304 (100%)
  R: 332/364 (91%)
  S: 391/490 (80%)
  T: 338/444 (76%)
  U: 309/309 (100%)
  V: 270/370 (73%)
  W: 213/312 (68%)
  X: 431/431 (100%)
  Y: 395/395 (100%)
  Z: 374/574 (65%)


In [23]:
import time

lote, _ = next(iter(test_generated))
una_imagen = lote[:1]   # batch de 1 imagen

# Calentamiento (las primeras inferencias son más lentas)
for _ in range(5):
    model_cargado.predict(una_imagen, verbose=0)

n = 100
inicio = time.time()
for _ in range(n):
    model_cargado.predict(una_imagen, verbose=0)
ms = (time.time() - inicio) / n * 1000
print(f"Tiempo por imagen: {ms:.2f} ms | {1000/ms:.1f} img/s")

Tiempo por imagen: 43.91 ms | 22.8 img/s


## Ronda 2 — FINE TUNING: carga del modelo, descongelado y recompilación

In [55]:
import tensorflow as tf
from tensorflow.keras.utils import image_dataset_from_directory

model_cargado = tf.keras.models.load_model('model_my_tf_fine_tuned_.keras')


base = model_cargado.layers[1]   # ajusta el índice al que viste en el print
print(f"La base tiene {len(base.layers)} capas internas")

La base tiene 157 capas internas


In [56]:
base.trainable = True

total_layers = len(base.layers)
limit = int(total_layers * 0.80)

for layer in base.layers[:limit]:
    layer.trainable = False

print(f"Congeladas: {limit} | Entrenables: {total_layers - limit}")

model_cargado.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy', 'precision', 'recall']
)


Congeladas: 125 | Entrenables: 32


## Carga de datos y entrenamiento

In [57]:
from pathlib import Path
DESTINO = Path('./asl_split_nuevo')   
train_dir = DESTINO / 'train'
test_dir = DESTINO / 'test'
val_dir = DESTINO / 'validate'



trained_generated = image_dataset_from_directory(
    train_dir,
    labels='inferred',
    batch_size = 100,
    color_mode='rgb',
    image_size=(224, 224),
    label_mode="categorical",
    pad_to_aspect_ratio=True,
)

test_generated = image_dataset_from_directory(
    test_dir,
    labels='inferred',
    batch_size = 100,
    color_mode='rgb',
    image_size=(224, 224),
    label_mode="categorical",
    pad_to_aspect_ratio=True,
)

val_generated = image_dataset_from_directory(
    val_dir,
    labels='inferred',
    batch_size = 100,
    color_mode='rgb',
    image_size=(224, 224),
    label_mode="categorical",
    pad_to_aspect_ratio=True,
)

earlyStopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True 
)

Found 8620 files belonging to 36 classes.
Found 14517 files belonging to 36 classes.
Found 12267 files belonging to 36 classes.


In [58]:
history_ft = model_cargado.fit(
    trained_generated,
    epochs=30,
    validation_data=val_generated,
    callbacks=[earlyStopping]
)

Epoch 1/30


/Users/denilsonbarahona/Documents/IA/model-sign-language/.venv/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


87/87 ━━━━━━━━━━━━━━━━━━━━ 172s 2s/step - accuracy: 0.8947 - loss: 0.4538 - precision: 0.9192 - recall: 0.8839 - val_accuracy: 0.7116 - val_loss: 1.0650 - val_precision: 0.7513 - val_recall: 0.6835
Epoch 2/30
87/87 ━━━━━━━━━━━━━━━━━━━━ 146s 2s/step - accuracy: 0.9898 - loss: 0.0399 - precision: 0.9925 - recall: 0.9867 - val_accuracy: 0.7280 - val_loss: 0.9814 - val_precision: 0.7660 - val_recall: 0.7081
Epoch 3/30
87/87 ━━━━━━━━━━━━━━━━━━━━ 161s 2s/step - accuracy: 0.9945 - loss: 0.0222 - precision: 0.9954 - recall: 0.9935 - val_accuracy: 0.7431 - val_loss: 0.9460 - val_precision: 0.7813 - val_recall: 0.7241
Epoch 4/30
87/87 ━━━━━━━━━━━━━━━━━━━━ 119s 1s/step - accuracy: 0.9971 - loss: 0.0145 - precision: 0.9976 - recall: 0.9965 - val_accuracy: 0.7682 - val_loss: 0.8857 - val_precision: 0.8023 - val_recall: 0.7475
Epoch 5/30
87/87 ━━━━━━━━━━━━━━━━━━━━ 114s 1s/step - accuracy: 0.9974 - loss: 0.0109 - precision: 0.9980 - recall: 0.9972 - val_accuracy: 0.7850 - val_loss: 0.8435 - val_preci

## Guardado del modelo final

In [59]:
model_cargado.save('finetunned.keras')

## Evaluación: matriz de confusión, classification report, aciertos por clase

In [64]:
print(trained_generated.class_names)

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']


In [41]:
model_cargado.evaluate(test_generated)

146/146 ━━━━━━━━━━━━━━━━━━━━ 69s 473ms/step - accuracy: 0.8737 - loss: 0.4810 - precision: 0.8873 - recall: 0.8673


[0.4809586703777313, 0.8737342357635498, 0.887315034866333, 0.8673279881477356]

In [35]:
import numpy as np

y_true, y_pred = [], []

for imagenes, etiquetas in test_generated:
    salidas = model_cargado.predict(imagenes, verbose=0)
    y_pred.extend(np.argmax(salidas, axis=1))     # clase predicha
    y_true.extend(np.argmax(etiquetas.numpy(), axis=1))  # one-hot -> índice

y_pred = np.array(y_pred)
y_true = np.array(y_true)
nombres_clases = test_generated.class_names
print(f"{len(y_pred)} predicciones recolectadas")

14517 predicciones recolectadas


In [36]:
from sklearn.metrics import confusion_matrix

matriz = confusion_matrix(y_true, y_pred)
num_clases = len(nombres_clases)

# Ancho de cada celda (sube a 5 si algún conteo pasa de 999)
w = 4

# Encabezado: nombres de las clases predichas, arriba
encabezado = " " * 6 + "".join(f"{c:>{w}}" for c in nombres_clases)
print("Predicho →")
print(encabezado)
print("Real ↓")

# Cada fila: nombre de la clase real + sus conteos
for i, clase_real in enumerate(nombres_clases):
    fila = f"{clase_real:>5} " + "".join(f"{int(matriz[i, j]):>{w}}" for j in range(num_clases))
    print(fila)

Predicho →
         0   1   2   3   4   5   6   7   8   9   A   B   C   D   E   F   G   H   I   J   K   L   M   N   O   P   Q   R   S   T   U   V   W   X   Y   Z
Real ↓
    0   60   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    1    0 368   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    2    0   2 306   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0  96   0   0   0   0
    3    0   0   2 359   0   0   0  25   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    4    0   0   0   0 418   0  12   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    5    0   0   0   0  52 471   0   0   0   5   0   0   0   0   0   0   0  

In [36]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred, target_names=nombres_clases, digits=3))

              precision    recall  f1-score   support

           0      0.952     1.000     0.976        60
           1      0.521     0.864     0.650       368
           2      0.784     0.755     0.769       404
           3      1.000     1.000     1.000       386
           4      0.955     0.830     0.888       430
           5      1.000     0.936     0.967       528
           6      0.664     0.833     0.739       413
           7      0.981     0.540     0.697       474
           8      0.721     0.982     0.831       382
           9      0.917     1.000     0.957       464
           A      1.000     1.000     1.000       453
           B      0.880     0.826     0.852       425
           C      1.000     0.839     0.912       453
           D      0.901     0.713     0.796       457
           E      0.996     0.609     0.756       427
           F      0.821     0.687     0.748       495
           G      0.997     0.777     0.873       448
           H      1.000    

In [38]:
print("ACIERTOS por clase:")
for i, clase in enumerate(nombres_clases):
    total = int(matriz[i, :].sum())
    ok = int(matriz[i, i])
    print(f"{clase:>3}: {ok:>3}/{total:<3} ({ok/total*100:.0f}%)")

ACIERTOS por clase:
  0:  60/60  (100%)
  1: 368/368 (100%)
  2: 306/404 (76%)
  3: 359/386 (93%)
  4: 418/430 (97%)
  5: 471/528 (89%)
  6: 403/413 (98%)
  7: 296/474 (62%)
  8: 344/382 (90%)
  9: 464/464 (100%)
  A: 453/453 (100%)
  B: 425/425 (100%)
  C: 380/453 (84%)
  D: 261/457 (57%)
  E: 334/427 (78%)
  F: 302/495 (61%)
  G: 351/448 (78%)
  H: 286/286 (100%)
  I: 359/479 (75%)
  J: 517/523 (99%)
  K: 265/346 (77%)
  L: 366/366 (100%)
  M: 330/330 (100%)
  N: 303/368 (82%)
  O: 468/480 (98%)
  P: 279/279 (100%)
  Q: 304/304 (100%)
  R: 293/364 (80%)
  S: 390/490 (80%)
  T: 344/444 (77%)
  U: 309/309 (100%)
  V: 286/370 (77%)
  W: 213/312 (68%)
  X: 417/431 (97%)
  Y: 395/395 (100%)
  Z: 450/574 (78%)
